In [2]:
import os
os.makedirs("outputs", exist_ok=True)

import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

# The standard device check — you'll use this pattern in every PyTorch notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

Using device: cpu
PyTorch version:     2.12.0
TorchVision version: 0.27.0


## PyTorch Tensors

### Tensor Question 1

In [3]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

b = torch.zeros(2, 3)
c = torch.ones(4)

print("Tensor a")
print(f"Value: {a}")
print(f"Shape: {a.shape}")
print(f"Dtype: {a.dtype}")
print(f"Device: {a.device}")

print("Tensor b")
print(f"Value: {b}")
print(f"Shape: {b.shape}")
print(f"Dtype: {b.dtype}")
print(f"Device: {b.device}")

print("Tensor c")
print(f"Value: {c}")
print(f"Shape: {c.shape}")
print(f"Dtype: {c.dtype}")
print(f"Device: {c.device}")

my_device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')
print('Torch device: {}'. format (my_device))

Tensor a
Value: tensor([[1., 2., 3.],
        [4., 5., 6.]])
Shape: torch.Size([2, 3])
Dtype: torch.float32
Device: cpu
Tensor b
Value: tensor([[0., 0., 0.],
        [0., 0., 0.]])
Shape: torch.Size([2, 3])
Dtype: torch.float32
Device: cpu
Tensor c
Value: tensor([1., 1., 1., 1.])
Shape: torch.Size([4])
Dtype: torch.float32
Device: cpu
Torch device: mps


Device these tensors are on right now is 'cpu', but I run a torch accelarator check and it is GPU ('cuda').

If I was running a training loop on the GPU, it matters that my model weights and my input tensors are on the same device, cause the code will throw a runtime error. This is in order to do any computation involving two or more tensors.

### Tensor Question 2

In [4]:
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])

# Compute and print the element-wise square root using torch.sqrt().
x_sqrt = torch.sqrt(x)
print(f"Element-wise square root:\n{x_sqrt}")
# Compute and print the sum using .sum().
x_sum = x.sum()
print(f"Sum of elements:\n{x_sum}")
# Compute and print the mean using .mean().
x_mean = x.mean()
print(f"Mean of elements:\n{x_mean}")
# Find and print the index of the maximum value using .argmax().
x_argmax = x.argmax()
print(f"Index of maximum value:\n{x_argmax}")

Element-wise square root:
tensor([1., 2., 3., 4., 5.])
Sum of elements:
55.0
Mean of elements:
11.0
Index of maximum value:
4


In the context of a classifier that outputs scores for 1,000 classes, .argmax() gives me the index of the class with the highest score. This index corresponds to the predicted class label for the input data.

### Tensor Question 3

In [5]:
a_gpu   = a.to(device)
print(f"a_gpu device: {a_gpu.device}")

a_back  = a_gpu.cpu()
a_numpy = a_back.numpy()
print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

a_gpu device: cpu
numpy type: <class 'numpy.ndarray'>
numpy values:
[[1. 2. 3.]
 [4. 5. 6.]]


PyTorch requires .cpu() before I can call .numpy(), cause NumPy arrays can only be created from CPU tensors
This tells me that NumPy arrays live in CPU, so if I need a tensor is on the GPU, it should be converted back to the CPU before a NumPy array

# Tensor Question 4

In [13]:
# Shape manipulation appears constantly when preparing images for neural networks. Starting from:

t = torch.arange(24).float()

# Write code to:
#
# Reshape t to (4, 6) and print the shape.
t_4_6 = t.reshape(4, 6)
print(f"Shape after reshaping to (4, 6): {t_4_6.shape}")
# Reshape t to (2, 3, 4) and print the shape.
t_2_3_4 = t.reshape(2, 3, 4)
print(f"Shape after reshaping to (2, 3, 4): {t_2_3_4.shape}")
# Take your result from step 1 and add a new dimension at position 0. Print the new shape.
t_4_6_new_dim = t_4_6.unsqueeze(0)
print(f"Shape after adding new dimension at position 0: {t_4_6_new_dim.shape}")

Shape after reshaping to (4, 6): torch.Size([4, 6])
Shape after reshaping to (2, 3, 4): torch.Size([2, 3, 4])
Shape after adding new dimension at position 0: torch.Size([1, 4, 6])


> Add a comment: a single image tensor typically has shape (channels, height, width). Neural networks expect batches with shape (batch_size, channels, height, width). What operation accomplishes this when you are processing one image at a time, and why does it matter?

- When processing one image at a time, the operation that accomplishes this is `unsqueeze(0)` which adds a new dimension at position 0. This transforms the shape from (channels, height, width) to (1, channels, height, width) creating a batch of size 1.
- This matters because neural networks are designed to process batches of data, and even when processing a single image, it needs to be in the correct shape to be compatible with the model's expected input format. By adding the batch dimension, we make sure that the image can be fed into the neural network without causing shape-related errors.

### Tensor Question 5

In [6]:
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])

t_a  = torch.tensor(np_a, dtype=torch.float32)
t_b  = torch.tensor(np_b, dtype=torch.float32)

# Compute the matrix product using NumPy and print the result.
np_product = np_a @ np_b
print(f"NumPy matrix product:\n{np_product}")
# Compute the same product using PyTorch and print the result.
torch_product = t_a @ t_b
print(f"PyTorch matrix product:\n{torch_product}")
# Confirm the outputs match.
assert np.allclose(np_product, torch_product.numpy()), "The outputs do not match"

NumPy matrix product:
[[19. 22.]
 [43. 50.]]
PyTorch matrix product:
tensor([[19., 22.],
        [43., 50.]])


At a high level matrix multiplication plays the role of transforming the input data and then result to be passed further to the next layer of a neural network.
The result of this operation is then passed through an activation function (to avoid linearity) and it helps each layer of neural network to learn.

## Pretrained Models

### Model Question 1

In [7]:
weights = ResNet18_Weights.DEFAULT
model   = models.resnet18(weights=weights)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters:     11,689,512
Trainable parameters: 11,689,512


> Add a comment: ResNet18 has roughly 11 million parameters.
Training it from scratch required approximately 1.2 million labeled ImageNet images and days of multi-GPU compute.
What does that tell you about the practical value of starting from pretrained weights when you're on a deadline or a budget?

Starting from pretrained weights helps a lot to not spend time and money on training the model by myself, especially with a large dataset like ImageNet (that you need to get from somewhere).



### Model Question 2

In [8]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

> Then answer the following in comments:
What is the name of the final layer in ResNet18, and what is its output size? (This number is the total count of ImageNet categories the model can predict.)
Can you identify the blocks named layer1 through layer4? These are the "deep" part of the network — the feature extractor. In plain terms, what does it mean for a network to be "deep"?

- The name of the final layer in ResNet18 is "fc", and its output size is 1000.
- I can. A network is been "deep" when it has many layers which helps it to capture complex patterns and representations in the data. That helps to increase performance with for example image classification.


### Model Question 3

In [9]:
model = model.to(device)
model.eval()
print("Model ready for inference.")

Model ready for inference.


> Add a comment explaining each line:
What does .to(device) do, and why does it need to match the device your input tensors will be on?
What does model.eval() change about the model's behavior? Name at least one layer type that behaves differently in training mode vs. evaluation mode.

- `.to(device)` moves the model's parameters and buffers to the given device (GPU for this case). It needs to match the device, cause both the model and the input tensors must be on the same device for computations to work correctly and to not get Error.
- `model.eval()` switches on an evaluation mode. It is done to disable layers like dropout and batch normalization training behavior.

### Model Question 4

In [10]:
# TorchVision model weights include the exact preprocessing pipeline the model was trained with. Use it directly:

preprocess = weights.transforms()
print(preprocess)

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


>This prints the full transform chain. Add a comment describing in plain English what each step does and why it matters. Address:
What does the resize/crop step accomplish?
What does ToTensor() do to the pixel value range?
What is normalization doing, and why does it use ImageNet's specific mean and standard deviation values rather than, say, mean=0.5, std=0.5?

- The resize/crop step resizes the input image to a size needed based on the model's architecture. This to be done to make sure all input images are of the same size to avoid model errors.
- The ToTensor() step converts the image from a PIL Image format to a PyTorch tensor and scales the pixel values from the range [0, 255] to [0.0, 1.0].
- Normalization is adjusting the pixel values so that they have a mean of 0 and a standard deviation of 1 based on the ImageNet dataset's specific mean and standard deviation values. This is cause the model was trained on specific dataset of ImageNet, and we need to be on the same page with the input data.

## Running Inference

In [11]:
# With a model loaded and preprocessing defined, running inference on a new image takes about five lines of code. This section walks you through each step using real images from the Intel Image Classification dataset.

# Add this image-loading helper to your notebook. It picks a random image from a given scene class:

import random
random.seed(42)

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path  = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name

# Get the ImageNet class labels from the weights metadata — no separate download needed:

imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")


Number of classes: 1000
First 5 labels: ['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead']


### Inference Question 1

In [12]:
# Write a function that runs inference on a single PIL image and returns the top-5 predicted class names and their probabilities.
# The function signature and steps are given below — fill in the implementation:

def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    Returns a list of (class_name, probability) tuples.
    """
    # Step 1: Preprocess the image and add a batch dimension
    # (hint: use preprocess(), .unsqueeze(0), and .to(device))
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    # Step 2: Run inference inside a torch.no_grad() block
    # (hint: call model() on your input tensor to get output of shape (1, 1000))
    with torch.no_grad():
        output = model(input_tensor)

    # Step 3: Convert raw scores (logits) to probabilities
    # (hint: use torch.nn.functional.softmax on output[0])
    probabilities = torch.nn.functional.softmax(output[0], dim=0)

    # Step 4: Get the top 5 predictions using torch.topk
    # (hint: returns top_probs and top_indices)
    top_probs, top_indices = torch.topk(probabilities, k=5)

    # Step 5: Build and return a list of (class_name, probability) tuples
    top5_predictions = [(class_labels[idx], prob.item()) for idx, prob in zip(top_indices, top_probs)]
    return top5_predictions

# Test it on one mountain image:

img, img_name = load_sample_image("mountain")
preds         = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

print(f"\nTop-5 predictions for '{img_name}':")
for class_name, prob in preds:
    print(f"  {class_name:30s}  {prob:.4f}")

IndexError: Cannot choose from an empty sequence

>Add a comment: does the top prediction make sense? Remember that the model was trained on ImageNet's 1,000 categories, which include things like "alp", "valley", and "lakeside" rather than simply "mountain". Do any of the top-5 labels map onto what you'd describe as a mountain scene?


The top prediction "alp" (0.4911) does make sense as it is a type of mountain. The other predictions like "volcano" and "valley" are also related as they are on the picture (at least the "valley"). The mountain itself is black on the pciture, so can be similar to how volcanos look like.

### Inference Question 2

In [ ]:
# Run inference on one image from each of the six scene classes. For each, print the top-3 predictions:

for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]
    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

>Add a comment: which classes does the model seem most confident about (high top-1 probability)? Which does it seem least confident about? Is there a pattern?

- The model seems most confident about the "mountain" class, with a top-1 probability of 0.5933 for "ski". Also, for "buildings" with a top-1 of 0.4301 for "palace".
- It seems least confident about the "street" class, with a top-1 probability of 0.1299 for "unicycle".
- A pattern is that the model is more confident about classes that a huge and/or distinct, and less confident about classes thatare broad and less distinct (like "street").

### Inference Question 3

In [ ]:
# The raw output of the model before softmax is called logits — unconstrained scores that can be any real number. After softmax they become probabilities that sum to 1. Observe the difference:

img, _ = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")

>Add a comment: why do neural networks output logits internally rather than probabilities? In a production pipeline that needs to filter out low-confidence predictions, which representation would you work with — logits or probabilities — and why?

- Neural networks output logits internally rather than probabilities cause logits can take on any real value which allows for more flexibility in the learning process.
- In a production pipeline that needs to filter out low-confidence predictions, I would work with probabilities instead of logits, cause probabilities are much easier to understand and interpret for both humans and machines.

### Inference Question 4

In [ ]:
# Create a visualization that shows an image alongside a horizontal bar chart of its top-5 predictions. Use plt.subplots(1, 2) with one panel for the image and one for the bar chart. Save it to outputs/warmup_inference_viz.png.
# You have all the pieces — img, preds, and plt — from the previous questions. Write the visualization yourself.
# img, img_name = load_sample_image("forest")
# preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img)
axes[0].axis("off")
axes[0].set_title(f"Input Image: {img_name}")

class_names = [p[0] for p in preds]
probs = [p[1] for p in preds]

bars = axes[1].barh(class_names[::-1], probs[::-1], color="steelblue")
axes[1].set_xlabel("Probability")
axes[1].set_title("Top-5 Predictions")
axes[1].set_xlim(0, 1)
for bar, prob in zip(bars, probs[::-1]):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                 f"{prob:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/warmup_inference_viz.png", dpi=150)
plt.show()

> Add a comment: how would you adapt this kind of visualization for a dashboard that a non-technical team member needs to review flagged predictions? What threshold on the top-1 probability might you use to decide when a prediction is "confident enough" to act on?

- For a non-technical dashboard, I would switch from probabilities to a more understandable self-descriptive English options (e.g., "High", "Medium", "Low"). Also, colors can help (e.g., "green", "yellow", "red"). and use color coding (green/yellow/red) to make confidence levels immediately intuitive. Maybe something around more descriptiove chart title with a simple verdict like "Confident" or "Needs Review", and a short description of what the model detected.
- For a threshold I would use 0.50 to start as a "confident enough" and then tune based on false positives vs. false negatives for the specific use case. This to be validated by humans, especially much lower than 0.50 values.
